### Q1. Display all columns and rows from the customers table.

In [ ]:
SELECT * FROM customers;

### Q2. Retrieve only first_name, last_name, and city of all customers.

In [ ]:
SELECT first_name, last_name, city
FROM customers;

### Q3. List all unique categories in the products table.

In [ ]:
SELECT DISTINCT category
FROM products;

### Q4. Identify the Primary Key of each table. Why must a PK be UNIQUE and NOT NULL?

| Table | Primary Key |
|-------------|-------------|
| customers | customer_id |
| products | product_id |
| orders | order_id |
| order_items | item_id |
  
Each row must be individually identifiable — two rows with the same PK would make it impossible to distinguish between them or reference them reliably.
A row without a primary key has no identity. The database can't join to it or reference it from other tables, making it a dangling, unreachable record.

### Q5. What constraints are on the email column? What happens on a duplicate insert?

```sql
email  VARCHAR(100)  UNIQUE  NOT NULL
```

- **UNIQUE** — no two rows can have the same email value.
- **NOT NULL** — every customer must have an email; blank is not allowed.

If you try to insert a duplicate email, MySQL throws:
> `Error 1062: Duplicate entry 'xyz@email.com' for key 'customers.email'`

The INSERT is rejected entirely — no row is written.

### Q6. Insert a product with unit_price = -50. What happens?

In [ ]:
INSERT INTO products (product_id, product_name, category, brand, unit_price, stock_qty)
VALUES (101, 'Laptop', 'Electronics', 'HP', -50, 10);

**Error received:**
`Error Code 3819: Check constraint 'products_chk_1' is violated.  (0.000 sec)`

The column definition `CHECK (unit_price > 0)` prevents any negative price from being stored. The statement fails immediately without inserting any row.

---
## Section B — Filtering & Optimization (WHERE, Indexes)

### Q7. Retrieve all orders with status = 'Delivered'.

In [ ]:
SELECT *
FROM orders
WHERE status = 'Delivered';

### Q8. Products in Electronics category with unit_price > ₹2000.

In [ ]:
SELECT *
FROM products
WHERE category = 'Electronics'
  AND unit_price > 2000;

### Q9. Customers who joined in 2024 and belong to Maharashtra.

In [ ]:
SELECT *
FROM customers
WHERE YEAR(join_date) = 2024
  AND state = 'Maharashtra';

### Q10. Orders between 2024-08-10 and 2024-08-25 (inclusive) that are NOT Cancelled.

In [ ]:
SELECT *
FROM orders
WHERE order_date BETWEEN '2024-08-10' AND '2024-08-25'
  AND status != 'Cancelled';

### Q11. Explain idx_orders_date and how it improves performance.

`idx_orders_date` is a B-Tree index on the `order_date` column of the `orders` table.  
Without it, MySQL has to scan every row (full table scan) to find matching dates.  
With the index, it can binary-search the sorted date values and jump straight to the matching rows — reducing work from O(n) to roughly O(log n).

It especially helps range queries like the one below:

In [ ]:
-- this query benefits from idx_orders_date
SELECT *
FROM orders
WHERE order_date BETWEEN '2025-01-01' AND '2025-01-31';

### Q12. Does `YEAR(join_date) = 2024` use the index? Rewrite to make it SARGable.

**No, the index is NOT used.**  
Wrapping `join_date` inside `YEAR()` applies a function to every single row before comparing — the database can't use the index because the index stores raw date values, not pre-computed year values.

*Index-friendly rewrite:*

In [ ]:
SELECT *
FROM customers
WHERE join_date >= '2024-01-01'
  AND join_date <  '2025-01-01';

---
## Section C — Aggregation (GROUP BY, SUM, COUNT, AVG, MIN, MAX)

### Q13. Count the total number of orders.

In [ ]:
SELECT COUNT(*) AS total_orders
FROM orders;

### Q14. Total revenue from all 'Delivered' orders.

In [ ]:
SELECT SUM(total_amount) AS total_revenue
FROM orders
WHERE status = 'Delivered';

### Q15. Average unit_price of products in each category.

In [ ]:
SELECT category,
       AVG(unit_price) AS avg_price
FROM products
GROUP BY category;

### Q16. Count of orders and total revenue per status, sorted by revenue descending.

In [ ]:
SELECT status,
       COUNT(*)          AS order_count,
       SUM(total_amount) AS revenue
FROM orders
GROUP BY status
ORDER BY revenue DESC;

### Q17. Most expensive and cheapest product in each category.

In [ ]:
SELECT category,
       MIN(unit_price) AS cheapest,
       MAX(unit_price) AS most_expensive
FROM products
GROUP BY category;

### Q18. Categories where average unit_price > ₹2000.  *(Hint: HAVING)*

In [ ]:
SELECT category,
       AVG(unit_price) AS avg_price
FROM products
GROUP BY category
HAVING AVG(unit_price) > 2000;

---
## Section D — Joins & Relationships

### Q19. INNER JOIN — each order with customer's first_name and last_name.

In [ ]:
SELECT o.order_id,
       o.order_date,
       c.first_name,
       c.last_name,
       o.total_amount
FROM orders o
INNER JOIN customers c ON o.customer_id = c.customer_id;

### Q20. LEFT JOIN — all customers and their orders (NULL for customers with no orders).

In [ ]:
SELECT c.first_name,
       c.last_name,
       o.order_id,
       o.total_amount
FROM customers c
LEFT JOIN orders o ON c.customer_id = o.customer_id;

### Q21. 3-table JOIN: orders → order_items → products.

In [ ]:
SELECT oi.order_id,
       p.product_name,
       oi.quantity,
       oi.unit_price,
       oi.discount_pct
FROM order_items oi
INNER JOIN orders   o ON oi.order_id   = o.order_id
INNER JOIN products p ON oi.product_id = p.product_id;

### Q22. Difference between LEFT JOIN, RIGHT JOIN, and FULL OUTER JOIN.

| Join Type | Rows Included | When to Use |
|---|---|---|
| **LEFT JOIN** | All rows from the left table + matched rows from the right (NULL where no match) | All customers, even those with no orders |
| **RIGHT JOIN** | All rows from the right table + matched rows from the left (NULL where no match) | All orders, even if customer record was deleted |
| **FULL OUTER JOIN** | All rows from both tables — NULL on whichever side has no match | Complete audit: see everything, matched or not |

**Example — LEFT JOIN:**

In [ ]:
-- customers who have never ordered will still show up (order columns = NULL)
SELECT c.first_name, o.order_id
FROM customers c
LEFT JOIN orders o ON c.customer_id = o.customer_id;

### Q23. Foreign Key relationships. What happens with customer_id = 999?

**Foreign Keys in this schema:**

| Table | FK Column | References |
|---|---|---|
| orders | customer_id | customers(customer_id) |
| order_items | order_id | orders(order_id) |
| order_items | product_id | products(product_id) |

**What happens if you insert an order with customer_id = 999?**

> `Error 1452: Cannot add or update a child row — a foreign key constraint fails.`

This is **Referential Integrity** at work. MySQL checks that customer_id 999 exists in the customers table before allowing the insert. Since it doesn't exist, the insert is rejected — preventing an orphan order record.

---
## Section E — Advanced Concepts (CASE, ACID, Transactions)

### Q24. Classify products into Budget / Mid-Range / Premium using CASE.

In [ ]:
SELECT product_name,
       unit_price,
       CASE
           WHEN unit_price < 1000               THEN 'Budget'
           WHEN unit_price BETWEEN 1000 AND 3000 THEN 'Mid-Range'
           ELSE                                      'Premium'
       END AS price_tier
FROM products;

### Q25. Count Delivered vs Not Delivered orders in a single row.

In [ ]:
SELECT
    SUM(CASE WHEN status = 'Delivered' THEN 1 ELSE 0 END) AS delivered,
    SUM(CASE WHEN status != 'Delivered' THEN 1 ELSE 0 END) AS not_delivered
FROM orders;

### Q26. Explain ACID with a bank transfer example.

**A — Atomicity**  
Every operation in a transaction is all-or-nothing. If any step fails, the entire thing is rolled back as if nothing happened.  
*Example:* Transferring ₹500 from Account A → B. If the debit from A succeeds but the credit to B crashes, Atomicity rolls back the debit too. The ₹500 neither disappears nor gets duplicated.

**C — Consistency**  
A transaction always moves the database from one valid state to another. All constraints, rules, and integrity checks must hold before and after.  
*Example:* If a bank rule says `balance >= 0`, Consistency ensures no transaction can leave any account with a negative balance, even if the SQL technically ran.

**I — Isolation**  
Concurrent transactions execute as if they were run one after another. Intermediate states are invisible to other transactions.  
*Example:* Two users simultaneously check Account A (₹1000) and both try to withdraw ₹800. Isolation ensures only one goes through — the second sees the updated balance before it commits.

**D — Durability**  
Once a transaction is committed, its changes are permanent — even a server crash immediately after cannot undo them. Data is written to disk (WAL / redo logs).  
*Example:* After you receive a bank transfer confirmation, a power outage shouldn't erase it. The committed record survives.

### Q27. Write a complete transaction: insert order 1011, two items, update stock. ROLLBACK on failure.

In [ ]:
START TRANSACTION;

-- Step 1: insert the new order
INSERT INTO orders (order_id, customer_id, order_date, status, total_amount)
VALUES (1011, 102, CURDATE(), 'Pending', 1598.00);

-- Step 2a: insert first order item  (Bedsheet Set — product 206)
INSERT INTO order_items (item_id, order_id, product_id, quantity, unit_price, discount_pct)
VALUES (5016, 1011, 206, 1, 1299.00, 0);

-- Step 2b: insert second order item  (Cushion Covers — product 208)
INSERT INTO order_items (item_id, order_id, product_id, quantity, unit_price, discount_pct)
VALUES (5017, 1011, 208, 1, 599.00, 0);

-- Step 3a: reduce stock for product 206
UPDATE products
SET stock_qty = stock_qty - 1
WHERE product_id = 206;

-- Step 3b: reduce stock for product 208
UPDATE products
SET stock_qty = stock_qty - 1
WHERE product_id = 208;

-- all steps succeeded -> make changes permanent
COMMIT;

-- if any step above fails, run this instead:
-- ROLLBACK;

**How ROLLBACK works in practice:**  
In application code (Python, Node.js, etc.) the above block sits inside a `try-catch`:  
- Any failed INSERT or UPDATE triggers `ROLLBACK` → database rewinds to exactly where it was before `START TRANSACTION`.  
- If all steps pass → `COMMIT` → changes are persisted permanently to disk.  

MySQL also rolls back automatically if the connection drops before a COMMIT — this is ACID Atomicity in action.

---
*End of Assignment*